# CloudGuard AWS configuration scanner

This notebook extends CPSA with a deterministic, read-only AWS configuration scanner. All cloud calls are handled by moto; no AWS account, credentials, Docker, LocalStack, or AI service is used. Run **Runtime > Run all** in Colab after uploading the project folder or zip.

In [ ]:
import glob, importlib, importlib.metadata as md, os, sys, zipfile
if not os.path.isdir('cloudguard'):
    for archive in glob.glob('cloud_password_analyzer*.zip'):
        zipfile.ZipFile(archive).extractall('.')
    if os.path.isdir('cloud_password_analyzer'):
        os.chdir('cloud_password_analyzer')
sys.path.insert(0, os.getcwd())
assert os.path.isdir('cloudguard') and os.path.isdir('cpsa')
!pip install -q boto3 'moto>=5' bcrypt argon2-cffi pandas pytest streamlit networkx

In [ ]:
required = {'boto3':'boto3', 'moto':'moto', 'pandas':'pandas', 'networkx':'networkx', 'streamlit':'streamlit'}
for module, distribution in required.items():
    importlib.import_module(module)
    print(distribution, md.version(distribution))
from cloudguard.simulation import CloudGuardSimulation
from cloudguard.engine import scan_account
print('CloudGuard imports verified')

## End-to-end insecure and secure scans

In [ ]:
with CloudGuardSimulation() as sim:
    sim.seed_cloudguard_account()
    insecure = scan_account(sim)
    print('Insecure:', insecure['score'])
with CloudGuardSimulation() as sim:
    sim.seed_cloudguard_account(secure=True)
    secure = scan_account(sim)
    print('Secure:', secure['score'])

In [ ]:
from cloudguard.reporting import findings_table
display(findings_table(insecure['findings']))

In [ ]:
from cloudguard.attack_paths import find_attack_paths
from cloudguard.remediation import apply_remediation
from cloudguard.faults import FaultInjector
with CloudGuardSimulation() as sim:
    sim.seed_cloudguard_account()
    before = scan_account(sim)
    print('Paths:', find_attack_paths(before['findings']))
    print('Fault:', scan_account(sim, FaultInjector(kind='access_denied', service='s3'))['score'])
    print('Remediation:', apply_remediation(sim)['fixed'])


## Optional Streamlit quick tunnel (Colab)

In [ ]:
# Optional Colab dashboard tunnel. Install cloudflared first if the binary is unavailable.
import shutil, subprocess, time
if not shutil.which('cloudflared'):
    print('Install cloudflared in Colab, then rerun this cell: https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/downloads/')
else:
    subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'])
    time.sleep(3)
    tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8501'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for _ in range(30):
        line = tunnel.stdout.readline()
        if 'trycloudflare.com' in line:
            print(line.strip())
            break

In [ ]:
!python -m pytest -q